In [1]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from mordred import Calculator, descriptors

## Prepare the data

In [8]:
INPUT_FILE      = r"C:\Users\hp\Desktop\stage etis\dataset\7Q27.signatures_with_smiles.csv"
OUTPUT_FILE     = r"C:\Users\hp\Desktop\stage etis\dataset\dataset_with_mordred.csv"
NAN_THRESHOLD   = 0.20   
CORR_THRESHOLD  = 0.95   
 

df = pd.read_csv(INPUT_FILE, sep=";")
df = df[df["SMILES"].notna() & (df["SMILES"].str.strip() != "")].reset_index(drop=True)
unique = df[["item", "SMILES"]].drop_duplicates(subset="SMILES").copy()


## Convert the smiles to 3d molecule to get the 3D descriptors

In [9]:
def smiles_to_3d_mol(smiles):
    
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    mol = Chem.AddHs(mol)                         
    result = AllChem.EmbedMolecule(mol, randomSeed=42)
    if result == -1:                               
        return None
    AllChem.MMFFOptimizeMolecule(mol)              
    mol = Chem.RemoveHs(mol)
    return mol

## Computing the mordred descriptors 

In [10]:
calc = Calculator(descriptors, ignore_3D=False)

mordred_rows = []
for _, row in unique.iterrows():
    mol = smiles_to_3d_mol(row["SMILES"])
    if mol is None:
        print(f"  WARNING: 3D embedding failed for {row['item']} → {row['SMILES']}")
        mordred_rows.append({"SMILES": row["SMILES"]})
        continue
    result = calc(mol)
    desc_dict = {str(k): v for k, v in result.items()}
    desc_dict["SMILES"] = row["SMILES"]
    mordred_rows.append(desc_dict)
 
mordred_df = pd.DataFrame(mordred_rows)

desc_cols = [c for c in mordred_df.columns if c != "SMILES"]
mordred_df[desc_cols] = mordred_df[desc_cols].apply(pd.to_numeric, errors="coerce")

## Filtering

### Dropping the descriptors with too many NAN values 

In [11]:
n_mols = len(mordred_df)
 

nan_frac = mordred_df[desc_cols].isna().mean()
keep = nan_frac[nan_frac <= NAN_THRESHOLD].index.tolist()
dropped_nan = len(desc_cols) - len(keep)
print(f"Dropped (>{NAN_THRESHOLD*100:.0f}% NaN):       {dropped_nan}")
desc_cols = keep
 
# for the ones left with little Nan values i replaced the Nan with the median 

mordred_df[desc_cols] = mordred_df[desc_cols].fillna(mordred_df[desc_cols].median())


Dropped (>20% NaN):       386


### Dropping the descriptors with 0 variance 

In [12]:
variance = mordred_df[desc_cols].var()
keep = variance[variance > 0].index.tolist()
dropped_var = len(desc_cols) - len(keep)
print(f"Dropped (zero variance):      {dropped_var}")
desc_cols = keep

Dropped (zero variance):      305


### Drop the highly correlated descriptor pairs 

In [13]:
corr_matrix = mordred_df[desc_cols].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > CORR_THRESHOLD)]
desc_cols = [c for c in desc_cols if c not in to_drop]
print(f"Dropped (corr > {CORR_THRESHOLD}):     {len(to_drop)}")
 
print(f"Final descriptor count:       {len(desc_cols)}")

Dropped (corr > 0.95):     1038
Final descriptor count:       97


In [14]:
mordred_clean = mordred_df[["SMILES"] + desc_cols]
result_df = df.merge(mordred_clean, on="SMILES", how="left")
result_df.to_csv(OUTPUT_FILE, index=False)
print(f"The file saved in {OUTPUT_FILE}")

The file saved in C:\Users\hp\Desktop\stage etis\dataset\dataset_with_mordred.csv
